# Data structures

This page provides complete documentation of the data structures to represent quantum algorithms in Qualtran. These classes are importable by the top-level `qualtran` namespace. They are re-exported from files within the `qualtran/_infra/*.py` subdirectory.

These data structures are used to represent quantum circuits, their underlying compute graphs, and the data types used within them.

### `Register`

The `Register` class represents a quantum register, which is a collection of qubits. It has the following attributes:

*   `name`: The name of the register.
*   `dtype`: The quantum data type of the register (e.g., `QBit`, `QUInt`, `QFxp`).
*   `shape`: The shape of the register (e.g., `(2, 3)` for a 2x3 register).
*   `side`: Whether the register is an input (`Side.LEFT`), output (`Side.RIGHT`), or both (`Side.THRU`).

The `total_bits` method returns the total number of qubits in the register, which is the product of `bitsize` and `shape`. The `adjoint` method returns a copy of the register with the `side` attribute flipped (i.e., `Side.LEFT` becomes `Side.RIGHT` and vice versa). 

Bloq authors will likely construct `Register` objects when declaring a bloq's signature via overriding the `Bloq.signature` abstract property.

In [ ]:
from qualtran import QBit, Register, Side

# Create a 2x3 register of qubits named 'my_reg'
reg = Register(name='my_reg', dtype=QBit(), shape=(2, 3), side=Side.LEFT)

# Get the total number of qubits in the register
total_qubits = reg.total_bits()

# Get the adjoint of the register
adj_reg = reg.adjoint()

### `Signature`

The `Signature` class represents the signature of a quantum bloq (an operation), which specifies the input and output registers. It is analogous to a function signature in classical programming. The `Signature` class ensures that each input and output register has a unique name. It provides methods for accessing the input registers (`lefts`), output registers (`rights`), and the total number of qubits in the signature (`n_qubits`). The `adjoint` method returns a copy of the signature with the input and output registers flipped. 

Bloq authors will likely construct a `Signature` object when declaring a bloq's signature via overriding the `Bloq.signature` abstract property.

In [ ]:
from qualtran import QAny, QBit, Register, Signature

# Create a signature with two input registers and one output register
sig = Signature(
    [
        Register(name='in_reg1', dtype=QAny(5), side=Side.LEFT),
        Register(name='in_reg2', dtype=QBit(), side=Side.LEFT),
        Register(name='out_reg', dtype=QAny(6), side=Side.RIGHT),
    ]
)

# Get the input registers
input_regs = sig.lefts()

# Get the output registers
output_regs = sig.rights()

# Get the total number of qubits in the signature
total_qubits = sig.n_qubits()

# Get the adjoint of the signature
adj_sig = sig.adjoint()

### `BloqInstance`

The `BloqInstance` class represents a specific instance of a bloq within a composite bloq (a bloq composed of other bloqs). It has two attributes:

*   `bloq`: The underlying `Bloq` object.
*   `i`: An index to disambiguate multiple instances of the same bloq.

The `bloq_is` method checks if the underlying bloq is an instance of a given type.

**Users should never directly construct a `BloqInstance`**. Construction of `BloqInstance`s is handled by `BloqBuilder`. Users or developers may sometimes access the `i` index of a bloq instance, but this index should never be assumed to have any semantic meaning.

In [ ]:
from qualtran import BloqInstance
from qualtran.bloqs.basic_gates import Hadamard

# Create an instance of the Hadamard bloq
# FOR DEMONSTRATION ONLY: BloqBuilder will construct BloqInstances for you.
hadamard_instance = BloqInstance(bloq=Hadamard(), i=0)

### `Soquet` and `Connection`

The `Soquet` class represents a connection point on a `BloqInstance`. It specifies a particular register and an optional index within that register. The `Connection` class represents a connection between two `Soquet` objects. It is used to define the data flow within a composite bloq.

**Users should never directly construct a `Soquet`**. Construction of `Soquet`s is handled by `BloqBuilder`. Users may sometimes access the properties of a `Soquet`.

In [ ]:
from qualtran import BloqInstance, QAny, Register, Soquet, Connection
from qualtran.bloqs.basic_gates import CNOT

# Create two soquets
# FOR DEMONSTRATION ONLY: BloqBuilder will construct these objects for you.
soq1 = Soquet(BloqInstance(CNOT(), i=0), Register('ctrl', QAny(1)))
soq2 = Soquet(BloqInstance(CNOT(), i=0), Register('target', QAny(1)))

# Create a connection between the two soquets
cxn = Connection(soq1, soq2)

### `Register` vs. `Soquet`

*   A `Register` represents a collection of qubits with a name, data type, shape, and direction (input, output, or both). It is a blueprint for an input/output of an operation. Bloq authors declare a bloq's registers.
*   A `Soquet` represents a specific connection point on a `BloqInstance`. It specifies a `Register` and an optional index within that register, acting like a handle or reference to a specific point within the quantum circuit's data flow. The framework will manage the construction and manipulation of soquets.

### `Bloq` vs. `BloqInstance`

*   A `Bloq` is a blueprint for a quantum operation, defining its signature and how it can be decomposed into sub-operations. Bloq authors write subclasses of `Bloq`. `Bloq` classes can have attributes (i.e. `__init__` args) to classically configure the quantum operation being represented. For example, the `TGate` class can make `TGate()` or `TGate(is_adjoint=True)` objects.
*   A `BloqInstance` is a specific instance of a `Bloq` object within a composite bloq. Multiple `BloqInstance` objects can be created from the same `Bloq` object, each with a unique index to distinguish them.

### Example: Two CNOT circuit

Each CNOT gate in a quantum circuit can be represented by the same `Bloq` object with the same `Signature` and `Registers`. However, when constructing a `CompositeBloq`, each CNOT gate is a distinct `BloqInstance` within the compute graph, allowing for precise connections. Two CNOTs joined control-to-control is a different circuit than joining control-to-target.

In [ ]:
# Each `Bloq` object follows value equality
cnot1 = CNOT()
cnot2 = CNOT()
assert cnot1 == cnot2

In [ ]:
# `BloqInstance` emulates reference equality
cnot_binst1 = BloqInstance(CNOT(), i=0)
cnot_binst2 = BloqInstance(CNOT(), i=1)
assert cnot_binst1 != cnot_binst2

### `BloqBuilder`

The `BloqBuilder` class is a mutable utility class for constructing composite bloqs. It provides methods for adding registers, adding bloq instances, and connecting them together. The `finalize` method returns an immutable `CompositeBloq` object.

Internally, it is a list of bloq instances and connections that can be appended to using the public methods. `BloqBuilder` will construct `Soquet` objects and `BloqInstance` objects to add a bloq to the compute graph.

### Manually composing bloqs

This code snippet below demonstrates how one could construct a `CompositeBloq` manually by creating the necessary components (`BloqInstance`, `Soquet`, `Connection`) and assembling them using the `CompositeBloq` constructor. **This approach is verbose and brittle.** Users should always use `BloqBuilder`. Nevertheless, the code snippet provides an explicit demonstration of the underlying structure of a `CompositeBloq`.

**Steps:**

1.  **Create Bloq Instances:** We create two instances of the `Hadamard()` bloq  using `BloqInstance`.
2.  **Define Registers:** We create `Register` objects in accordance with our desired signature and the signature of the bloqs we're composing.
3.  **Create Soquets:** We create `Soquet` objects to represent the connection points for each (bloq instance, register) tuple. We also must create sentinel connection points for the dangling inputs and outputs of the composite bloq.
5.  **Define Connections:** We define the connections between soquets using `Connection` objects.
6.  **Construct CompositeBloq:** Finally, we construct the `CompositeBloq` object using the signature, bloq instances, and connections.



In [ ]:
from qualtran import (
    Bloq, BloqInstance, CompositeBloq,
    Connection, Register, Side, Signature,
    Soquet, LeftDangle, RightDangle,
)
from qualtran.bloqs.basic_gates import CNOT, Hadamard

# Construct a composite bloq "by hand"
# DO NOT DO THIS: Use BloqBuilder to avoid creating an invalid graph
hadamard1 = BloqInstance(Hadamard(), i=0)
hadamard2 = BloqInstance(Hadamard(), i=1)

my_reg = Register(name="my_reg", dtype=QBit())  # can name whatever we like
hadamard_reg = Register(name="q", dtype=QBit()) # Must match Hadamard().signature

left_soq = Soquet(binst=LeftDangle, reg=my_reg)
hadamard_soq_1 = Soquet(binst=hadamard1, reg=hadamard_reg)
hadamard_soq_2 = Soquet(binst=hadamard2, reg=hadamard_reg)
right_soq = Soquet(binst=RightDangle, reg=my_reg)

connections = [
    Connection(left_soq, hadamard_soq_1),
    Connection(hadamard_soq_1, hadamard_soq_2),
    Connection(hadamard_soq_2, right_soq),
]

composite_bloq = CompositeBloq(
    signature=Signature(registers=[my_reg]),
    bloq_instances=[hadamard1, hadamard2],
    connections=connections,
)

from qualtran.drawing import show_bloq
show_bloq(composite_bloq)